** Data Preprocessing**

*My Steps -> Summary*
- Merge: all 3 CSVs
- Keep: useful columns
- Remove: missing values
- Create: sentiment labels
- Clean: review text
- Remove: very short reviews
- Save: final dataset

In [ ]:
#Imports

import pandas as pd
from pathlib import Path
import re

In [ ]:
#Merging the csv data
data_folder = Path("/content")
csv_files = list(data_folder.glob("*.csv"))
csv_files

#reading each CSV safely
for file in csv_files:
    temp_df = pd.read_csv(
        file,
        engine="python",
        on_bad_lines="skip"
    )

    df_list.append(temp_df)

#merging all CSV files into one dataframe
df = pd.concat(df_list, ignore_index=True)

#checking merged dataset size
df.shape

(67992, 27)

In [ ]:
#Reviewing data
df = df [[
    "name",
    "reviews.rating",
    "reviews.text"
]]

df.head()

#Removing rows where at least one of the three columns is empty
df = df.dropna(subset=["name", "reviews.rating", "reviews.text"])
df.shape

(61199, 3)

In [ ]:
#Converting the ratings to integers
df["reviews.rating"] = df["reviews.rating"].astype(int)
df["reviews.rating"].unique()

#Creating sentiment column
def map_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"


df["sentiment"] = df["reviews.rating"].apply(map_sentiment)
df["sentiment"].value_counts()

#Outcome: Many positive reviews. Pretty imbalanced


#Cleaning the text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ",text) #remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s]", " ",text) # remove special characters, keep letters, numbers, spaces only.
    text = re.sub(r"\s+", " ",text).strip() #remove the extra spaces too

    return text


#applying funciton to every review
df["clean_review"] = df["reviews.text"].apply(clean_text)

#showing original vs cleaned text
df[["reviews.text", "clean_review"]].head()


#showing full text side by side (no truncation)
pd.set_option('display.max_colwidth', None)
df[["reviews.text", "clean_review"]].sample(10)

#removing short reviews like "good", "ok", etc.
df = df[df["clean_review"].str.len() > 10]
df.shape

(59900, 5)

In [ ]:
#Saving the cleaned dataset

#defining path for cleaned file to be saved to
output_path = Path("/content/outputs/clean_reviews.csv")
output_path.parent.mkdir(exist_ok=True)

#saving the df as a csv
df.to_csv(output_path, index=False)

#printing the condirmation for double-check
print("Saved to:", output_path)

Saved to: /content/outputs/clean_reviews.csv
